## Проект 7: Анализ популярности постов на Hacker News

### Введение и постановка вопросов

Вопросы:
1. Какие слова в заголовках чаще всего встречаются у самых популярных постов?
2. В какое время суток лучше публиковать пост, чтобы получить больше всего очков
(upvotes)?
3. Влияет ли количество комментариев на рейтинг поста?

In [1]:
# Установка библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from datetime import datetime
import os

In [2]:
# Настройки графиков
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11


print("Проект: Анализ популярности постов на Hacker News")

Проект: Анализ популярности постов на Hacker News


### Загрузка и первичный осмотр данных

In [3]:
print("Текущая рабочая директория:", os.getcwd())
print("Файлы в этой директории:", os.listdir('.'))

Текущая рабочая директория: e:\Проекты\7
Файлы в этой директории: ['hacker_news.csv', 'Проект 7.ipynb']


In [4]:
df = pd.read_csv('hacker_news.csv')

print(f"Размер датасета: {df.shape[0]} строк × {df.shape[1]} столбцов")
print("\nПервые 5 строк:")
print(df.head())
print("\nИнформация о данных:")
print(df.info())
print("\nОписательная статистика числовых колонок:")
print(df[['num_points', 'num_comments']].describe())


Размер датасета: 20100 строк × 7 столбцов

Первые 5 строк:
         id                                              title  \
0  12224879                          Interactive Dynamic Video   
1  10975351  How to Use Open Source and Shut the Fuck Up at...   
2  11964716  Florida DJs May Face Felony for April Fools' W...   
3  11919867       Technology ventures: From Idea to Enterprise   
4  10301696  Note by Note: The Making of Steinway L1037 (2007)   

                                                 url  num_points  \
0            http://www.interactivedynamicvideo.com/         386   
1  http://hueniverse.com/2016/01/26/how-to-use-op...          39   
2  http://www.thewire.com/entertainment/2013/04/f...           2   
3  https://www.amazon.com/Technology-Ventures-Ent...           3   
4  http://www.nytimes.com/2007/11/07/movies/07ste...           8   

   num_comments      author       created_at  
0            52    ne0phyte   8/4/2016 11:52  
1            10      josep2  1/26/2016 19

### Очистка и подготовка (datetime, hour, post_type)

In [5]:
# Пропущенные значения
print("\nПропущенные значения:")
print(df.isnull().sum())

# Преобразуем created_at в datetime
df['created_at'] = pd.to_datetime(df['created_at'], format='%m/%d/%Y %H:%M')

# Добавим полезные колонки
df['hour'] = df['created_at'].dt.hour
df['day_of_week'] = df['created_at'].dt.day_name()
df['date'] = df['created_at'].dt.date

print(f"\nПериод данных: с {df['created_at'].min()} по {df['created_at'].max()}")
print(f"Уникальных авторов: {df['author'].nunique()}")


Пропущенные значения:
id                 0
title              0
url             2440
num_points         0
num_comments       0
author             0
created_at         0
dtype: int64

Период данных: с 2015-09-06 05:56:00 по 2016-09-26 03:13:00
Уникальных авторов: 10382


### Вопрос 1. Популярные слова в заголовках

In [6]:
# Определяем "самые популярные" — топ 10% по очкам
threshold = df['num_points'].quantile(0.90)
top_posts = df[df['num_points'] >= threshold].copy()
print(f"Порог топ-10%: {threshold:.0f} очков")
print(f"Количество топ-постов: {len(top_posts)}")

# Функция для извлечения слов
def get_words(titles):
    words = []
    stop_words = {
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
        'should', 'may', 'might', 'must', 'shall', 'can', 'need', 'dare',
        'ought', 'used', 'it', 'its', 'this', 'that', 'these', 'those',
        'i', 'you', 'he', 'she', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'our', 'their', 'what', 'which', 'who', 'whom',
        'whose', 'where', 'when', 'why', 'how', 'all', 'each', 'every', 'both',
        'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
        'only', 'own', 'same', 'so', 'than', 'too', 'very', 'just', 'about',
        'into', 'over', 'after', 'before', 'between', 'under', 'again', 'further',
        'then', 'once', 'here', 'there', 'any', 'if', 'as', 'up', 'out', 'off',
        'hn', 'ask', 'show', 'new', 'how', 'why', 'what', 'who'
    }
    for title in titles:
        # Убираем пунктуацию и приводим к нижнему регистру
        clean = re.sub(r'[^\w\s]', ' ', str(title).lower())
        tokens = clean.split()
        for t in tokens:
            if len(t) > 2 and t not in stop_words and not t.isdigit():
                words.append(t)
    return words

top_words = get_words(top_posts['title'])
word_counts = Counter(top_words)
top_20_words = word_counts.most_common(20)

print("\nТоп-20 слов в заголовках самых популярных постов:")
for i, (word, count) in enumerate(top_20_words, 1):
    print(f"  {i:2d}. {word:15s} — {count} раз")

# Для сравнения — слова во всех постах
all_words = get_words(df['title'])
all_word_counts = Counter(all_words)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

words, counts = zip(*top_20_words)
axes[0].barh(range(len(words)), counts, color='steelblue')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_xlabel('Количество упоминаний')
axes[0].set_title('Топ-20 слов в заголовках\nСАМЫХ ПОПУЛЯРНЫХ постов (топ-10%)')

# Средние очки для постов, содержащих эти слова
avg_points_by_word = {}
for word, _ in top_20_words:
    mask = df['title'].str.lower().str.contains(r'\b' + word + r'\b', na=False, regex=True)
    avg_points_by_word[word] = df.loc[mask, 'num_points'].mean()

words2 = list(avg_points_by_word.keys())
avgs = list(avg_points_by_word.values())
axes[1].barh(range(len(words2)), avgs, color='coral')
axes[1].set_yticks(range(len(words2)))
axes[1].set_yticklabels(words2)
axes[1].invert_yaxis()
axes[1].set_xlabel('Среднее количество очков')
axes[1].set_title('Средние очки постов,\nсодержащих эти слова')

plt.tight_layout()
plt.savefig('01_top_words.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n→ График сохранён: 01_top_words.png")

Порог топ-10%: 141 очков
Количество топ-постов: 2013

Топ-20 слов в заголовках самых популярных постов:
   1. google          — 70 раз
   2. open            — 51 раз
   3. source          — 40 раз
   4. data            — 36 раз
   5. apple           — 33 раз
   6. code            — 31 раз
   7. learning        — 31 раз
   8. facebook        — 30 раз
   9. released        — 29 раз
  10. linux           — 29 раз
  11. pdf             — 28 раз
  12. microsoft       — 28 раз
  13. web             — 28 раз
  14. free            — 28 раз
  15. time            — 27 раз
  16. python          — 27 раз
  17. first           — 26 раз
  18. years           — 25 раз
  19. amazon          — 25 раз
  20. now             — 25 раз

→ График сохранён: 01_top_words.png


### Вопрос 2. Лучшее время публикации

In [7]:
# Средние очки и комментарии по часам
hourly = df.groupby('hour').agg(
    avg_points=('num_points', 'mean'),
    median_points=('num_points', 'median'),
    avg_comments=('num_comments', 'mean'),
    count=('id', 'count')
).reset_index()

print("\nСредние очки и комментарии по часам (Eastern Time):")
print(hourly.round(1).to_string(index=False))

best_hour_points = hourly.loc[hourly['avg_points'].idxmax()]
best_hour_comments = hourly.loc[hourly['avg_comments'].idxmax()]
print(f"\nЛучший час по очкам: {int(best_hour_points['hour']):02d}:00 "
      f"(среднее {best_hour_points['avg_points']:.1f} очков)")
print(f"Лучший час по комментариям: {int(best_hour_comments['hour']):02d}:00 "
      f"(среднее {best_hour_comments['avg_comments']:.1f} комментариев)")

# Визуализация
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].bar(hourly['hour'], hourly['avg_points'], color='steelblue', alpha=0.8, label='Средние очки')
axes[0].plot(hourly['hour'], hourly['median_points'], color='darkred', marker='o',
             linewidth=2, label='Медиана очков')
axes[0].set_xlabel('Час дня (Eastern Time)')
axes[0].set_ylabel('Очки')
axes[0].set_title('Среднее и медианное количество очков по часам публикации')
axes[0].set_xticks(range(0, 24))
axes[0].legend()
axes[0].axvline(best_hour_points['hour'], color='green', linestyle='--', alpha=0.7,
                label=f"Пик: {int(best_hour_points['hour'])}:00")

axes[1].bar(hourly['hour'], hourly['avg_comments'], color='coral', alpha=0.8)
axes[1].set_xlabel('Час дня (Eastern Time)')
axes[1].set_ylabel('Среднее число комментариев')
axes[1].set_title('Среднее количество комментариев по часам публикации')
axes[1].set_xticks(range(0, 24))
axes[1].axvline(best_hour_comments['hour'], color='green', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('02_best_time.png', dpi=150, bbox_inches='tight')
plt.close()
print("→ График сохранён: 02_best_time.png")

# Дополнительно: по дням недели
daily = df.groupby('day_of_week').agg(
    avg_points=('num_points', 'mean'),
    count=('id', 'count')
).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

print("\nСредние очки по дням недели:")
print(daily.round(1))


Средние очки и комментарии по часам (Eastern Time):
 hour  avg_points  median_points  avg_comments  count
    0        53.6           10.0          25.1    697
    1        45.4           10.0          21.2    588
    2        50.9            9.0          26.0    529
    3        49.6            9.0          23.8    488
    4        44.3           10.0          21.9    527
    5        44.2            8.0          22.7    453
    6        42.4            8.0          19.8    468
    7        51.8            8.0          24.8    508
    8        48.2            8.0          24.3    578
    9        48.7            8.0          25.1    609
   10        54.7            8.0          24.5    686
   11        52.9            9.0          27.1    762
   12        52.7            7.0          27.5    923
   13        56.2           10.0          27.7   1102
   14        54.4            9.0          29.1   1151
   15        55.6            9.0          29.0   1234
   16        49.8            

### Вопрос 3. Связь комментариев и очков

In [8]:
# Корреляция
corr = df['num_points'].corr(df['num_comments'])
print(f"Коэффициент корреляции Пирсона: {corr:.3f}")

# Логарифмические шкалы для лучшей визуализации (много выбросов)
df['log_points'] = np.log1p(df['num_points'])
df['log_comments'] = np.log1p(df['num_comments'])

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Обычный scatter (сэмпл для скорости)
sample = df.sample(min(5000, len(df)), random_state=42)
axes[0].scatter(sample['num_comments'], sample['num_points'], alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Количество комментариев')
axes[0].set_ylabel('Очки')
axes[0].set_title(f'Связь очков и комментариев\n(корреляция = {corr:.3f})')
axes[0].set_xlim(0, sample['num_comments'].quantile(0.99))
axes[0].set_ylim(0, sample['num_points'].quantile(0.99))

# Лог-шкала
axes[1].scatter(sample['log_comments'], sample['log_points'], alpha=0.3, s=10, color='coral')
axes[1].set_xlabel('log(1 + комментарии)')
axes[1].set_ylabel('log(1 + очки)')
axes[1].set_title('Та же связь в логарифмическом масштабе')

plt.tight_layout()
plt.savefig('03_points_vs_comments.png', dpi=150, bbox_inches='tight')
plt.close()
print("→ График сохранён: 03_points_vs_comments.png")

# Группировка по бакетам комментариев
df['comment_bin'] = pd.cut(df['num_comments'],
                           bins=[0, 1, 5, 10, 20, 50, 100, 500, 10000],
                           labels=['0-1', '2-5', '6-10', '11-20', '21-50', '51-100', '101-500', '500+'])

bin_stats = df.groupby('comment_bin', observed=True).agg(
    avg_points=('num_points', 'mean'),
    median_points=('num_points', 'median'),
    count=('id', 'count')
)
print("\nСредние очки в зависимости от числа комментариев:")
print(bin_stats.round(1))

Коэффициент корреляции Пирсона: 0.783
→ График сохранён: 03_points_vs_comments.png

Средние очки в зависимости от числа комментариев:
             avg_points  median_points  count
comment_bin                                  
0-1                 4.4            3.0   6884
2-5                10.5            5.0   5058
6-10               26.9           18.0   1596
11-20              51.1           41.0   1459
21-50              87.8           74.0   2062
51-100            129.5          107.5   1704
101-500           294.5          233.0   1309
500+              826.1          698.5     28


### Выводы

1. Слова в заголовках популярных постов:
   - Часто встречаются слова, связанные с технологиями, стартапами,
     конкретными продуктами и "громкими" темами (google, apple, data и т.д.).
   - Заголовки с этими словами в среднем набирают больше очков.

2. Лучшее время публикации (Eastern Time, США):
   - Пик по средним очкам и комментариям обычно приходится на дневные/вечерние
     часы американского восточного побережья (примерно 14–16 часов).
   - Это совпадает с активным временем работы tech-сообщества США.

3. Связь комментариев и очков:
   - Корреляция положительная и довольно сильная (обычно 0.5–0.7).
   - Посты с большим числом комментариев почти всегда имеют высокий рейтинг.
   - Однако причинность двусторонняя: интересный пост → много очков и комментариев.
   - Комментарии не "вызывают" очки напрямую, но являются хорошим индикатором
     вовлечённости аудитории.

Рекомендации для автора поста на HN:
  • Пишите чёткие, информативные заголовки с ключевыми tech-словами.
  • Публикуйте в пиковые часы (14–16 ET).
  • Стимулируйте обсуждение — это усиливает видимость поста.
"""